In [56]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [57]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [58]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)
tabla_servicios = pd.read_sql_table('mensajeria_servicio', mensajeria)
tabla_servicios = tabla_servicios[tabla_servicios['descripcion_cancelado'].isnull()]
tabla_servicios["key_trans_servicio"] = range(1,len(tabla_servicios)+1 )
tabla_servicios.drop(columns=['novedades', 'fecha_deseada', 'hora_deseada','nombre_recibe','telefono_recibe','hora_visto_por_mensajero',
                              'descripcion_pago','ida_y_regreso','tipo_pago_id','prioridad','visto_por_mensajero','multiples_origenes',
                              'descripcion_cancelado','descripcion_multiples_origenes','activo','asignar_mensajero','es_prueba',
                              'nombre_solicitante','descripcion','origen_id','tipo_vehiculo_id'], inplace=True)

tabla_servicios['mensajero_id']=tabla_servicios['mensajero_id'].fillna(0).astype(int)
tabla_servicios['mensajero_id']=tabla_servicios['mensajero2_id'].fillna(0).astype(int)
tabla_servicios['mensajero_id']=tabla_servicios['mensajero3_id'].fillna(0).astype(int)
tabla_servicios.info()

<class 'pandas.core.frame.DataFrame'>
Index: 346 entries, 465 to 28409
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id                  346 non-null    int64         
 1   fecha_solicitud     346 non-null    datetime64[ns]
 2   hora_solicitud      346 non-null    object        
 3   cliente_id          346 non-null    int64         
 4   destino_id          346 non-null    int64         
 5   mensajero_id        346 non-null    int64         
 6   tipo_servicio_id    346 non-null    int64         
 7   usuario_id          346 non-null    int64         
 8   ciudad_destino_id   346 non-null    int64         
 9   ciudad_origen_id    346 non-null    int64         
 10  mensajero2_id       72 non-null     float64       
 11  mensajero3_id       10 non-null     float64       
 12  key_trans_servicio  346 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(9), object(

In [59]:
tabla_servicios.head(10)

,id,fecha_solicitud,hora_solicitud,cliente_id,destino_id,mensajero_id,tipo_servicio_id,usuario_id,ciudad_destino_id,ciudad_origen_id,mensajero2_id,mensajero3_id,key_trans_servicio
465,510,2024-02-03,09:37:18,5,214,0,3,7,1,1,NaN,NaN,1
808,794,2024-02-08,14:15:40,5,214,0,3,7,1,1,NaN,NaN,2
811,791,2024-02-08,12:07:55,5,214,0,3,11,1,1,NaN,NaN,3
822,154,2024-01-25,08:45:15,5,214,0,3,7,1,1,NaN,NaN,4
918,16686,2024-06-12,09:00:31,11,4,0,2,160,1,6,NaN,NaN,5
1048,3970,2024-03-11,11:28:52,2,401,0,1,173,1,1,NaN,NaN,6
1105,1090,2024-02-13,14:32:52,5,486,0,3,12,1,1,NaN,NaN,7
1270,1240,2024-02-15,09:28:43,7,346,0,3,188,1,1,NaN,NaN,8
1342,1647,2024-02-20,06:35:00,5,214,0,3,9,1,1,NaN,NaN,9
1509,2269,2024-02-26,02:57:54,2,401,0,3,173,1,1,NaN,NaN,10


In [60]:
tabla_servicios.to_sql('trans_servicio', etl_conn, if_exists='replace', index=False)

346